In [1]:
from megatron.bridge.models.gemma import gemma4_provider, modeling_gemma4 as gm
from megatron.bridge.recipes.gemma import gemma4_e4b_pretrain_config
from megatron.bridge.training.mixed_precision import get_mixed_precision_config
from megatron.core import parallel_state, tensor_parallel
from megatron.core.transformer.dot_product_attention import DotProductAttention
from megatron.core.transformer.module import Float16Module

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1016 17:25:09.692000 921626 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1016 17:25:09.723000 921626 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an o

ERROR 10-16 17:25:23 [config.py:29] Failed to import Triton kernels. Please make sure your triton version is compatible. Error: No module named 'triton_kernels.matmul_ogs'
LANG_RL Log directory: None


/opt/Megatron-Bridge/3rdparty/Megatron-LM/megatron/core/models/mimo/config/base_configs.py:12: UserWarning: MimoModelConfig is experimental and still under active development. The API may change without notice in future releases.
  class MimoModelConfig:


In [2]:
import torch

torch.distributed.init_process_group(
        "nccl", init_method="tcp://127.0.0.1:29791", world_size=1, rank=0
    )

In [3]:
parallel_state.initialize_model_parallel(1, 1)
tensor_parallel.model_parallel_cuda_manual_seed(1234)

In [4]:
patched = [
        (gm, "_gemma4_rms_norm"),
        (gm, "_logit_softcapping"),
        (gm.Gemma4DenseSelfAttention, "_v_norm"),
        (DotProductAttention, "forward"),
        (gemma4_provider.Gemma4DenseProvider, "build"),
    ]

In [5]:
patched

[(<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
  '_gemma4_rms_norm'),
 (<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
  '_logit_softcapping'),
 (megatron.bridge.models.gemma.modeling_gemma4.Gemma4DenseSelfAttention,
  '_v_norm'),
 (megatron.core.transformer.dot_product_attention.DotProductAttention,
  'forward'),
 (megatron.bridge.models.gemma.gemma4_provider.Gemma4DenseProvider, 'build')]

In [6]:
ootb = {key: getattr(*key) for key in patched}
ootb

{(<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
  '_gemma4_rms_norm'): <function megatron.bridge.models.gemma.modeling_gemma4._gemma4_rms_norm(hidden_states: torch.Tensor, weight: torch.Tensor | None, eps: float) -> torch.Tensor>,
 (<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
  '_logit_softcapping'): <function megatron.bridge.models.gemma.modeling_gemma4._logit_softcapping(logits: torch.Tensor, scale: float | None) -> torch.Tensor>,
 (megatron.bridge.models.gemma.modeling_gemma4.Gemma4DenseSelfAttention,
  '_v_norm'): <function megatron.bridge.models.gemma.modeling_gemma4.Gemma4DenseSelfAttention._v_norm(self, value: torch.Tensor) -> torch.Tensor>,
 (megatron.core.transformer.dot_product_attention.DotProductAttention,
  'forward'): <function megatron.core.transformer.dot_product_attention.DotProductAttent

In [7]:
import runpy
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent / "bridge/models/gemma4"))
import train

In [8]:
runpy.run_path = (
    lambda *args, **kwargs: None
)  # train.main() ends by starting training; stop it there
train.main()

[gemma4-fast] RMSNorm + value norm: torch.compile
[gemma4-fast] final logit softcapping: torch.compile
[gemma4-fast] attention: sliding layers fa2, global layers te


In [9]:
gm

<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>

In [10]:
ours = {key: getattr(*key) for key in patched}
ours

{(<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
  '_gemma4_rms_norm'): <function megatron.bridge.models.gemma.modeling_gemma4._gemma4_rms_norm(hidden_states: torch.Tensor, weight: torch.Tensor | None, eps: float) -> torch.Tensor>,
 (<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
  '_logit_softcapping'): <function megatron.bridge.models.gemma.modeling_gemma4._logit_softcapping(logits: torch.Tensor, scale: float | None) -> torch.Tensor>,
 (megatron.bridge.models.gemma.modeling_gemma4.Gemma4DenseSelfAttention,
  '_v_norm'): <function train.patch_norms.<locals>.<lambda>(self, value)>,
 (megatron.core.transformer.dot_product_attention.DotProductAttention,
  'forward'): <function train.patch_attention.<locals>.forward(self, query, key, value, attention_mask, attn_mask_type=None, attention_bias=None, packed_seq_par

In [11]:
k = list(ootb.keys())[0]
print(ours[k])
print(ootb[k])

print(ours[k] is ootb[k])
print(ours[k].__wrapped__ is ootb[k])

<function _gemma4_rms_norm at 0x7ffa8bca0040>
<function _gemma4_rms_norm at 0x7ffa8ca562a0>
False
True


In [12]:
cfg = gemma4_e4b_pretrain_config()
cfg

ConfigContainer(env_vars={'NCCL_GRAPH_REGISTER': 0, 'NCCL_NVLS_ENABLE': 0, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'TORCH_NCCL_AVOID_RECORD_STREAMS': 1, 'TORCH_NCCL_HIGH_PRIORITY': 1}, rng=RNGConfig(seed=1234, te_rng_tracker=False, inference_rng_tracker=False, data_parallel_random_init=False), rerun_state_machine=RerunStateMachineConfig(error_injection_rate=0, error_injection_type='transient_error', rerun_mode='disabled', check_for_nan_in_loss=True, check_for_spiky_loss=False, spiky_loss_factor=10.0), train=TrainingConfig(micro_batch_size=1, global_batch_size=8, rampup_batch_size=None, step_batch_size_schedule=None, decrease_batch_size_if_needed=False, empty_unused_memory_level=0, check_weight_hash_across_dp_replicas_interval=None, gpu_sniff_test_interval=None, train_sync_interval=None, train_iters=1000, train_samples=None, exit_interval=None, exit_duration_in_mins=None, exit_signal_handler=False, exit_signal=<Signals.SIGTERM: 15>, exit_signal_handler_for_dataloader=Fals

In [13]:
cfg.mixed_precision

'bf16_mixed'

In [14]:
get_mixed_precision_config(cfg.mixed_precision)

MixedPrecisionConfig(fp32=False, fp16=False, bf16=True, params_dtype=torch.bfloat16, pipeline_dtype=torch.bfloat16, autocast_dtype=None, autocast_enabled=False, grad_reduce_in_fp32=True, fp8=None, fp8_recipe='tensorwise', first_last_layers_bf16=False, fp8_margin=0, fp8_amax_history_len=1, fp8_amax_compute_algo='most_recent', fp8_wgrad=True, fp8_dot_product_attention=False, fp8_multi_head_attention=False, fp8_param=False, fp8_param_gather=False, fp4=None, fp4_recipe='nvfp4', fp4_param=False, fp4_param_gather=False, loss_scale=None, initial_loss_scale=4294967296, min_loss_scale=1.0, loss_scale_window=1000, hysteresis=2, num_layers_at_start_in_bf16=0, num_layers_at_end_in_bf16=0, reuse_grad_buf_for_mxfp8_param_ag=False)

In [15]:
cfg.model

Gemma4DenseProvider(tensor_model_parallel_size=2, tensor_parallel_num_weight_shards=None, gtp_weight_remat_size=1, pipeline_model_parallel_comm_backend=None, pipeline_model_parallel_size=1, virtual_pipeline_model_parallel_size=None, sequence_parallel=False, context_parallel_size=1, hierarchical_context_parallel_sizes=None, max_seqlen_per_dp_cp_rank=None, hybrid_context_parallel=False, expert_model_parallel_size=1, expert_tensor_parallel_size=None, expert_tensor_parallel_num_weight_shards=None, expert_gtp_weight_remat_size=1, perform_initialization=True, use_cpu_initialization=False, fp16=False, bf16=True, params_dtype=torch.bfloat16, timers=None, finalize_model_grads_func=None, grad_scale_func=None, moe_grad_scale_func=None, mtp_grad_scale_func=None, no_sync_func=None, grad_sync_func=None, param_sync_func=None, deterministic_mode=False, enable_autocast=False, autocast_dtype=torch.bfloat16, num_microbatches_with_partial_activation_checkpoints=None, gradient_accumulation_fusion=False, us

In [16]:
cfg.model.tensor_model_parallel_size

2

In [17]:
cfg.model.seq_length

4096

In [18]:
get_mixed_precision_config(cfg.mixed_precision).setup(cfg.model)
provider = cfg.model

In [19]:
SEQ = 4096

provider.tensor_model_parallel_size, provider.seq_length = 1, SEQ
provider.finalize()
model = (
    provider.provide().cuda()
)  # the per-layer-embedding modules are built on the CPU, as in training

/opt/Megatron-Bridge/3rdparty/Megatron-LM/megatron/core/transformer/transformer_config.py:2415: UserWarning: full scope is deprecated. Use empty cuda_graph_modules to capture the whole layer.
  warnings.warn(


[gemma4-fast] froze 18 unused k_layernorm weights in KV-shared layers


In [20]:
wrapped = Float16Module(provider, model)

In [21]:
from collections import Counter

def signature(v):
    return tuple(v.shape) if torch.is_tensor(v) else type(v).__name__

def spy(shapes, original):
    def recorded(*args, **kwargs):
        shapes[tuple(signature(a) for a in args)] += 1
        return original(*args, **kwargs)
    return recorded

values = {}
for key, original in ootb.items():
    values[key] = Counter()
    setattr(*key, spy(values[key], original))

tokens = torch.randint(0, 262144, (1, SEQ), device="cuda")
with torch.no_grad():
    wrapped(tokens, torch.arange(SEQ, device="cuda")[None], None)

for (owner, name), shapes in values.items():
    print(f"{owner.__name__}.{name}", dict(shapes))

megatron.bridge.models.gemma.modeling_gemma4._gemma4_rms_norm {((1, 4096, 42, 256), (256,), 'float'): 1, ((4096, 1, 2560), (2560,), 'float'): 210, ((4096, 1, 8, 256), (256,), 'float'): 35, ((4096, 1, 2, 256), (256,), 'float'): 35, ((4096, 1, 8, 512), (512,), 'float'): 7, ((4096, 1, 2, 512), (512,), 'float'): 7}
megatron.bridge.models.gemma.modeling_gemma4._logit_softcapping {((4096, 1, 262144), 'float'): 1}
Gemma4DenseSelfAttention._v_norm {('Gemma4DenseSelfAttention', (4096, 1, 2, 256)): 20, ('Gemma4DenseSelfAttention', (4096, 1, 2, 512)): 4}
DotProductAttention.forward {('DotProductAttention', (4096, 1, 8, 256), (4096, 1, 2, 256), (4096, 1, 2, 256), 'NoneType'): 35, ('DotProductAttention', (4096, 1, 8, 512), (4096, 1, 2, 512), (4096, 1, 2, 512), 'NoneType'): 7}
Gemma4DenseProvider.build {}


In [22]:
print(provider.hidden_size, provider.kv_channels, provider.global_kv_channels)

2560 256 512


In [23]:
from collections import Counter

def signature(v):
    return tuple(v.shape) if torch.is_tensor(v) else type(v).__name__

def spy(value, count, original):
    def recorded(*args, **kwargs):
        key = tuple(signature(a) for a in args)
        count[key] += 1

        if count[key] == 1:
            # value.append(list(args) + [kwargs] if kwargs else list(args))
            value.append(list(args) + [kwargs])
        return original(*args, **kwargs)
    return recorded

count = {}
value = {}
for key, original in ootb.items():
    count[key] = Counter()
    value[key] = list()
    setattr(*key, spy(value[key], count[key], original))

tokens = torch.randint(0, 262144, (1, SEQ), device="cuda")
with torch.no_grad():
    wrapped(tokens, torch.arange(SEQ, device="cuda")[None], None)

In [24]:
for key in ootb:
    for args in value[key]:
        args, kwargs = args[:-1], args[-1]
        out = ootb[key](*args, **kwargs)
        print(out.shape)

torch.Size([1, 4096, 42, 256])
torch.Size([4096, 1, 2560])
torch.Size([4096, 1, 8, 256])
torch.Size([4096, 1, 2, 256])
torch.Size([4096, 1, 8, 512])
torch.Size([4096, 1, 2, 512])
torch.Size([4096, 1, 262144])
torch.Size([4096, 1, 2, 256])
torch.Size([4096, 1, 2, 512])
torch.Size([4096, 1, 2048])
torch.Size([4096, 1, 4096])


# Tap 1

In [39]:
def run(function, input_args, input_kwargs, grad):
    """function's output and the gradients of the inputs that need one."""
    leaves_args = [
        x.detach().requires_grad_(x.requires_grad) if torch.is_tensor(x) else x
        for x in input_args
    ]
    leaves_kwargs = {
        k: v.detach().requires_grad_(v.requires_grad) if torch.is_tensor(v) else v
        for k, v in input_kwargs.items()
    }

    out = function(*leaves_args, **leaves_kwargs)
    out.backward(grad.to(out.dtype))
    return [out.detach()] + [
        x.grad for x in leaves_args if torch.is_tensor(x) and x.requires_grad
    ] + [v.grad for k, v in leaves_kwargs.items() if torch.is_tensor(v) and v.requires_grad]


def compare(name, ootb, ours, input_args, input_kwargs, labels):
    """Relative L2 error of each output, OOTB's and ours, against OOTB's code in float64."""
    input_args = [arg.bfloat16().requires_grad_() if torch.is_tensor(arg) else arg for arg in input_args]
    input_kwargs = {arg: value.bfloat16().requires_grad_() if torch.is_tensor(value) else value for arg, value in input_kwargs.items()}
    
    with torch.no_grad():
        probe = ootb(*input_args, **input_kwargs)
    grad = torch.randn(probe.shape, device="cuda", dtype=probe.dtype)
    del probe

    exact_input_args = [
        x.double() if torch.is_tensor(x) and x.is_floating_point() else x
        for x in input_args
    ]
    exact_input_kwargs = {
        k: v.double() if torch.is_tensor(v) and v.is_floating_point() else v
        for k, v in input_kwargs.items()
    }
    
    exact, base, mine = (
        run(f, args, kwargs, grad)
        for f, args, kwargs in (
            (ootb, exact_input_args, exact_input_kwargs), 
            (ootb, input_args, input_kwargs), 
            (ours, input_args, input_kwargs))
    )
    
    rel = lambda got, ref: (
        (got.double() - ref).norm() / ref.norm()
    ).item()  # noqa: E731
    rows = [
        dict(output=label, ootb=rel(b, e), ours=rel(m, e))
        for label, e, b, m in zip(labels, exact, base, mine)
    ]
    
    result = dict(
        kernel=name,
        identical=torch.equal(base[0], mine[0]),
        outputs=rows,
        passed=all(r["ours"] <= MARGIN * r["ootb"] + 1e-6 for r in rows),
    )
    print(
        f"{name:40s} {'PASS' if result['passed'] else 'FAIL'}  "
        + "  ".join(f"{r['output']} {r['ootb']:.2e} -> {r['ours']:.2e}" for r in rows),
        flush=True,
    )
    return result

In [40]:
rms_key = list(ootb.keys())[0]

In [41]:
args = value[rms_key][0]
input_args, input_kwargs = args[:-1], args[-1]

In [42]:
MARGIN = 1.25
compare("RMSNorm", ootb[rms_key], ours[rms_key], input_args, input_kwargs, ["out", "dx", "dweight"])

RMSNorm                                  PASS  out 1.66e-03 -> 1.66e-03  dx 2.33e-03 -> 1.66e-03  dweight 1.88e-03 -> 1.88e-03


{'kernel': 'RMSNorm',
 'identical': False,
 'outputs': [{'output': 'out',
   'ootb': 0.0016613981652734537,
   'ours': 0.0016613981664402415},
  {'output': 'dx', 'ootb': 0.002333823953340758, 'ours': 0.001655405105727208},
  {'output': 'dweight',
   'ootb': 0.0018778483597451455,
   'ours': 0.0018778483597451455}],
 'passed': True}

# Tap 2

In [72]:
import inspect
import os

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from megatron.bridge import AutoBridge
from transformers import AutoTokenizer


# Load model
# the -it checkpoint is multimodal; convert its text model only
HF = "google/gemma-4-E4B-it"
os.environ["GEMMA4_CONVERSION_MODE"] = "text"  
AutoBridge.from_hf_pretrained(HF).load_hf_weights([model])

# load dataset and tokenize
parquet = hf_hub_download(
    "HuggingFaceFW/fineweb-edu", "sample/10BT/000_00000.parquet",
    repo_type="dataset", revision="87f09149ef4734204d70ed1d046ddc9ca3f2b8f9",
)
texts = next(pq.ParquetFile(parquet).iter_batches(batch_size=64, columns=["text"])).column("text").to_pylist()
ids = AutoTokenizer.from_pretrained(HF)("\n\n".join(texts))["input_ids"][:SEQ]

if len(ids) < SEQ:
    raise RuntimeError(f"only {len(ids)} tokens of text; need {SEQ}")
tokens = torch.tensor([ids], device="cuda")

def signature(v):
    return tuple(v.shape) if torch.is_tensor(v) else type(v).__name__

# Tap the model at the changed functions and get the inputs.
# We will use these inputs to check "ours" vs "ootb"!
def spy(calls, original):
    sig = inspect.signature(original)
    def recorded(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        shape = tuple(signature(v) for v in bound.arguments.values())
        
        if shape not in calls:
            bound.arguments = {
                n: v.detach().clone().requires_grad_(v.is_floating_point()) if torch.is_tensor(v) else v
                for n, v in bound.arguments.items()
            }
            calls[shape] = bound
        
        return original(*args, **kwargs)
    return recorded

calls = {key: {} for key in ootb}
installed = {key: getattr(*key) for key in ootb} # For reinstating the original function
for key, original in ootb.items():
    setattr(*key, spy(calls[key], original))
    
try:
    with torch.no_grad():
        wrapped(tokens, torch.arange(SEQ, device="cuda")[None], None)
finally:
    # reinstate here
    for key, function in installed.items():
        setattr(*key, function)

/opt/venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [73]:
calls

{(<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
  '_gemma4_rms_norm'): {((1, 4096, 42, 256),
   (256,),
   'float'): <BoundArguments (hidden_states=tensor([[[[-2.2095e-02, -1.2891e-01,  1.6968e-02,  ...,  2.4048e-02,
             -3.4180e-02, -1.0864e-02],
            [-1.2268e-02, -3.6865e-02,  5.7373e-02,  ..., -7.1289e-02,
              6.2500e-02,  4.4678e-02],
            [ 2.7832e-02, -6.5918e-03,  6.6895e-02,  ...,  4.0039e-02,
             -1.0014e-04,  3.7994e-03],
            ...,
            [ 1.8555e-02,  2.1362e-02, -2.5269e-02,  ...,  1.7944e-02,
              3.7994e-03,  2.6703e-03],
            [-1.7944e-02,  1.7929e-03,  1.5335e-03,  ...,  4.4823e-04,
             -2.3315e-02,  1.5747e-02],
            [-4.9805e-02, -1.0498e-02, -2.7588e-02,  ...,  1.7456e-02,
             -1.8921e-02, -1.4496e-03]],
  
           [[-1.7090e-02, -5.9326e-02, -4.6631e-02,  ...,  5.0293e-02,
     

In [75]:
rms_key = list(ootb.keys())[0]
rms_key

(<module 'megatron.bridge.models.gemma.modeling_gemma4' from '/opt/Megatron-Bridge/src/megatron/bridge/models/gemma/modeling_gemma4.py'>,
 '_gemma4_rms_norm')

In [84]:
bound = next(iter(calls[rms_key].items()))[1]
bound

<BoundArguments (hidden_states=tensor([[[[-2.2095e-02, -1.2891e-01,  1.6968e-02,  ...,  2.4048e-02,
           -3.4180e-02, -1.0864e-02],
          [-1.2268e-02, -3.6865e-02,  5.7373e-02,  ..., -7.1289e-02,
            6.2500e-02,  4.4678e-02],
          [ 2.7832e-02, -6.5918e-03,  6.6895e-02,  ...,  4.0039e-02,
           -1.0014e-04,  3.7994e-03],
          ...,
          [ 1.8555e-02,  2.1362e-02, -2.5269e-02,  ...,  1.7944e-02,
            3.7994e-03,  2.6703e-03],
          [-1.7944e-02,  1.7929e-03,  1.5335e-03,  ...,  4.4823e-04,
           -2.3315e-02,  1.5747e-02],
          [-4.9805e-02, -1.0498e-02, -2.7588e-02,  ...,  1.7456e-02,
           -1.8921e-02, -1.4496e-03]],

         [[-1.7090e-02, -5.9326e-02, -4.6631e-02,  ...,  5.0293e-02,
            5.0049e-02, -4.1260e-02],
          [-1.0742e-01, -3.7231e-03,  5.2490e-02,  ..., -5.8594e-02,
            9.7046e-03,  5.1514e-02],
          [-1.6357e-02,  2.8931e-02,  1.9043e-02,  ...,  2.8076e-02,
           -4.0527e-02,  1.

In [94]:
def run(function, inputs, grad):
    """function's output and the gradients of the inputs that need one."""
    leaves = {
        k: v.detach().requires_grad_(v.requires_grad) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }
    out = function(**leaves)
    out.backward(grad.to(out.dtype))
    return [out.detach()] + [
        x.grad for k, x in leaves.items() if torch.is_tensor(x) and x.requires_grad
    ]

def compare(name, ootb, ours, bound):
    """Relative L2 error of each output, OOTB's and ours, against OOTB's code in float64."""
    sig, arguments = bound.signature, bound.arguments
    labels = ["out"] + [f"d{n}" for n, v in arguments.items() if torch.is_tensor(v) and v.requires_grad]
    
    with torch.no_grad():
        probe = ootb(**arguments)
    grad = torch.randn(probe.shape, device="cuda", dtype=probe.dtype)
    del probe

    exact_arguments = {
        k: v.double() if torch.is_tensor(v) and v.is_floating_point() else v
        for k, v in arguments.items()
    }
    
    exact, base, mine = (
        run(f, args, grad)
        for f, args in (
            (ootb, exact_arguments), 
            (ootb, arguments), 
            (ours, arguments))
    )
    
    rel = lambda got, ref: (
        (got.double() - ref).norm() / ref.norm()
    ).item()  # noqa: E731
    rows = [
        dict(output=label, ootb=rel(b, e), ours=rel(m, e))
        for label, e, b, m in zip(labels, exact, base, mine)
    ]
    
    result = dict(
        kernel=name,
        identical=torch.equal(base[0], mine[0]),
        outputs=rows,
        passed=all(r["ours"] <= MARGIN * r["ootb"] + 1e-6 for r in rows),
    )
    print(
        f"{name:40s} {'PASS' if result['passed'] else 'FAIL'}  "
        + "  ".join(f"{r['output']} {r['ootb']:.2e} -> {r['ours']:.2e}" for r in rows),
        flush=True,
    )
    return result

In [95]:
MARGIN = 1.25
compare("RMSNorm", ootb[rms_key], ours[rms_key], bound)

RMSNorm                                  PASS  out 1.60e-03 -> 1.60e-03  dhidden_states 2.26e-03 -> 1.66e-03  dweight 1.71e-03 -> 1.71e-03


{'kernel': 'RMSNorm',
 'identical': False,
 'outputs': [{'output': 'out',
   'ootb': 0.0015964957816050683,
   'ours': 0.0015964957819088808},
  {'output': 'dhidden_states',
   'ootb': 0.0022602416536057855,
   'ours': 0.001657823881176417},
  {'output': 'dweight',
   'ootb': 0.001712158072976676,
   'ours': 0.001712158072976676}],
 'passed': True}

In [86]:
bound.signature

<Signature (hidden_states: torch.Tensor, weight: torch.Tensor | None, eps: float) -> torch.Tensor>

In [82]:
bound.arguments

{'hidden_states': tensor([[[[-2.2095e-02, -1.2891e-01,  1.6968e-02,  ...,  2.4048e-02,
            -3.4180e-02, -1.0864e-02],
           [-1.2268e-02, -3.6865e-02,  5.7373e-02,  ..., -7.1289e-02,
             6.2500e-02,  4.4678e-02],
           [ 2.7832e-02, -6.5918e-03,  6.6895e-02,  ...,  4.0039e-02,
            -1.0014e-04,  3.7994e-03],
           ...,
           [ 1.8555e-02,  2.1362e-02, -2.5269e-02,  ...,  1.7944e-02,
             3.7994e-03,  2.6703e-03],
           [-1.7944e-02,  1.7929e-03,  1.5335e-03,  ...,  4.4823e-04,
            -2.3315e-02,  1.5747e-02],
           [-4.9805e-02, -1.0498e-02, -2.7588e-02,  ...,  1.7456e-02,
            -1.8921e-02, -1.4496e-03]],
 
          [[-1.7090e-02, -5.9326e-02, -4.6631e-02,  ...,  5.0293e-02,
             5.0049e-02, -4.1260e-02],
           [-1.0742e-01, -3.7231e-03,  5.2490e-02,  ..., -5.8594e-02,
             9.7046e-03,  5.1514e-02],
           [-1.6357e-02,  2.8931e-02,  1.9043e-02,  ...,  2.8076e-02,
            -4.0527e-0

# Final Script

In [1]:
#!/usr/bin/env python3
"""Functional check, kernel level: every kernel our changes swap in computes the same function as the one Megatron-Bridge
runs out of the box (OOTB), forward and backward, and is at least as accurate.

Each kernel runs on the same BF16 inputs as the OOTB one it replaces, and both are compared with the exact result: the
OOTB code itself run in float64, so every detail of Megatron's semantics (mask, window, softmax scale, GQA layout) is in
the reference. The inputs are the ones each patched function receives in one forward pass of the pretrained model on
FineWeb-Edu text, one call per distinct shape. The check also builds the model with train.py's patches and runs one
forward and backward: the parameters left without a gradient must be exactly the ones train.py freezes, with and
without the patches.

Runs in the container on one GPU; bridge/validate/run.py starts it and reads the JSON it writes.
"""

import argparse
import copy
import json
import runpy
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path.cwd().parent.parent / "bridge/models/gemma4"))
import train  # noqa: E402  the code under test
import inspect
import os

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from megatron.bridge import AutoBridge
from transformers import AutoTokenizer

SEQ = 1024
MARGIN = 1.25  # ours may be up to 25% less accurate than OOTB: a different summation order rounds differently


HF = "google/gemma-4-E4B-it"


def text_tokens():
    """The first SEQ tokens of FineWeb-Edu text, from the shard prepare.py pins."""
    parquet = hf_hub_download(
        "HuggingFaceFW/fineweb-edu", "sample/10BT/000_00000.parquet",
        repo_type="dataset", revision="87f09149ef4734204d70ed1d046ddc9ca3f2b8f9",
    )
    texts = next(pq.ParquetFile(parquet).iter_batches(batch_size=64, columns=["text"])).column("text").to_pylist()
    ids = AutoTokenizer.from_pretrained(HF)("\n\n".join(texts))["input_ids"][:SEQ]
    if len(ids) < SEQ:
        raise RuntimeError(f"only {len(ids)} tokens of text; need {SEQ}")
    return torch.tensor([ids], device="cuda")

# Get "ootb" and "ours" functions
def setup():
    """One-rank Megatron; the OOTB functions and train.py's replacements; the model as training builds it (random
    weights, TP 1) with train.py's patches applied. main() loads the pretrained weights after the gradient check."""
    from megatron.bridge.models.gemma import gemma4_provider, modeling_gemma4 as gm
    from megatron.bridge.recipes.gemma import gemma4_e4b_pretrain_config
    from megatron.bridge.training.mixed_precision import get_mixed_precision_config
    from megatron.core import parallel_state, tensor_parallel
    from megatron.core.transformer.dot_product_attention import DotProductAttention
    from megatron.core.transformer.module import Float16Module

    torch.distributed.init_process_group(
        "nccl", init_method="tcp://127.0.0.1:29791", world_size=1, rank=0
    )
    parallel_state.initialize_model_parallel(1, 1)
    tensor_parallel.model_parallel_cuda_manual_seed(1234)

    patched = [
        (gm, "_gemma4_rms_norm"),
        (gm, "_logit_softcapping"),
        (gm.Gemma4DenseSelfAttention, "_v_norm"),
        (DotProductAttention, "forward"),
        (gemma4_provider.Gemma4DenseProvider, "build"),
    ]
    ootb = {key: getattr(*key) for key in patched}
    runpy.run_path = (
        lambda *args, **kwargs: None
    )  # train.main() ends by starting training; stop it there
    train.main()
    ours = {key: getattr(*key) for key in patched}

    cfg = gemma4_e4b_pretrain_config()
    get_mixed_precision_config(cfg.mixed_precision).setup(cfg.model)
    provider = cfg.model
    provider.tensor_model_parallel_size, provider.seq_length = 1, SEQ
    provider.finalize()
    model = (
        provider.provide().cuda()
    )  # the per-layer-embedding modules are built on the CPU, as in training
    return provider, model, Float16Module(provider, model), ootb, ours


def params_without_grad(wrapped, functions):
    """Names of the parameters that get no gradient from one forward and backward on a random sequence."""
    for (owner, name), function in functions.items():
        setattr(owner, name, function)
    tokens = torch.randint(
        0,
        262144,
        (1, SEQ + 1),
        device="cuda",
        generator=torch.Generator("cuda").manual_seed(0),
    )
    logits = wrapped(tokens[:, :-1], torch.arange(SEQ, device="cuda")[None], None)
    F.cross_entropy(logits.flatten(0, 1).float(), tokens[:, 1:].flatten()).backward()
    names = {
        name.removeprefix("module.")
        for name, p in wrapped.named_parameters()
        if p.grad is None
    }
    wrapped.zero_grad(set_to_none=True)
    return names


def run(function, inputs, grad):
    """function's output and the gradients of the inputs that need one."""
    leaves = {
        k: v.detach().requires_grad_(v.requires_grad) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }
    out = function(**leaves)
    out.backward(grad.to(out.dtype))
    grads = {k: x.grad for k, x in leaves.items() if torch.is_tensor(x) and x.requires_grad}
    missing = [k for k, g in grads.items() if g is None]
    if missing:
        raise RuntimeError(f"no gradient for {missing}")
    return [out.detach()] + list(grads.values())

def compare(name, ootb, ours, bound):
    """Relative L2 error of each output, OOTB's and ours, against OOTB's code in float64."""
    arguments = bound.arguments
    labels = ["out"] + [f"d{n}" for n, v in arguments.items() if torch.is_tensor(v) and v.requires_grad]
    
    with torch.no_grad():
        probe = ootb(**arguments)
    grad = torch.randn(probe.shape, device="cuda", dtype=probe.dtype)
    del probe

    exact_arguments = {
        k: v.double() if torch.is_tensor(v) and v.is_floating_point() else v
        for k, v in arguments.items()
    }
    
    exact, base, mine = (
        run(f, args, grad)
        for f, args in (
            (ootb, exact_arguments), 
            (ootb, arguments), 
            (ours, arguments))
    )
    
    rel = lambda got, ref: (
        (got.double() - ref).norm() / ref.norm()
    ).item()  # noqa: E731
    rows = [
        dict(output=label, ootb=rel(b, e), ours=rel(m, e))
        for label, e, b, m in zip(labels, exact, base, mine)
    ]
    
    result = dict(
        kernel=name,
        identical=torch.equal(base[0], mine[0]),
        outputs=rows,
        passed=all(r["ours"] <= MARGIN * r["ootb"] + 1e-6 for r in rows),
    )
    print(
        f"{name:40s} {'PASS' if result['passed'] else 'FAIL'}  "
        + "  ".join(f"{r['output']} {r['ootb']:.2e} -> {r['ours']:.2e}" for r in rows),
        flush=True,
    )
    return result

# Helper function to get the shape of the input tensor
def signature(v):
    return tuple(v.shape) if torch.is_tensor(v) else type(v).__name__

# Tap the model at the changed functions and get the inputs.
# We will use these inputs to check "ours" vs "ootb"!
def spy(calls, original):
    sig = inspect.signature(original)
    def recorded(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        shape = tuple(signature(v) for v in bound.arguments.values())
        
        if shape not in calls:
            bound.arguments = {
                n: v.detach().clone().requires_grad_(v.requires_grad) if torch.is_tensor(v) else v
                for n, v in bound.arguments.items()
            }
            calls[shape] = bound
        
        return original(*args, **kwargs)
    return recorded

def main():
    from megatron.bridge.models.gemma import modeling_gemma4 as gm
    from megatron.core.transformer import attention
    from megatron.core.transformer.dot_product_attention import DotProductAttention

    provider, model, wrapped, ootb, ours = setup()
    frozen = {name for name, p in model.named_parameters() if not p.requires_grad}
    for p in model.parameters():
        p.requires_grad_(True)
    no_grad = {
        "OOTB": params_without_grad(wrapped, ootb),
        "ours": params_without_grad(wrapped, ours),
    }
    gradients = dict(
        frozen=sorted(frozen),
        **{k: sorted(v) for k, v in no_grad.items()},
        passed=frozen == no_grad["OOTB"] == no_grad["ours"],
    )
    print(
        f"{len(frozen)} parameters frozen; without a gradient: OOTB {len(no_grad['OOTB'])}, ours "
        f"{len(no_grad['ours'])}: {'PASS' if gradients['passed'] else 'FAIL'}",
        flush=True,
    )

    # the -it checkpoint is multimodal; convert its text model only
    os.environ["GEMMA4_CONVERSION_MODE"] = "text"  
    AutoBridge.from_hf_pretrained(HF).load_hf_weights([model])
    tokens = text_tokens()
    torch.manual_seed(0)

    # Fused RoPE is a config flag, not a patched function: ours is the same call with run.sh's
    # model.apply_rope_fusion=true config.
    fused = copy.copy(provider)
    fused.apply_rope_fusion = True
    rope = (attention, "apply_rotary_pos_emb")
    ootb[rope] = attention.apply_rotary_pos_emb
    ours[rope] = lambda **kwargs: ootb[rope](**{**kwargs, "config": fused})

    calls = {key: {} for key in ootb}
    installed = {key: getattr(*key) for key in ootb}  # train.py's functions, put back after the capture
    for key, original in ootb.items():
        setattr(*key, spy(calls[key], original))
        
    # try:
    #     # with autograd on, so each captured tensor needs a gradient exactly where training computes one
    #     logits = wrapped(tokens, torch.arange(SEQ, device="cuda")[None], None)
    #     # the KV-shared layers keep the last K/V (_stored_kv), and with it this pass's whole graph; backward frees it
    #     logits.sum().backward()
    #     del logits
    #     wrapped.zero_grad(set_to_none=True)
    # finally:
    #     # reinstate the original function
    #     for key, function in installed.items():
    #         setattr(*key, function)
    try:
        # autograd builds the graph, so each captured tensor needs a gradient exactly where training computes one,
        # but saves nothing: this pass never runs a backward
        with torch.autograd.graph.saved_tensors_hooks(lambda t: None, lambda _: None):
            wrapped(tokens, torch.arange(SEQ, device="cuda")[None], None)
    finally:
        for key, function in installed.items():
            setattr(*key, function)

    torch.cuda.empty_cache()  # the softcap check's 8.6 GB float64 tensors don't fit the blocks cached so far
    results = []
    for key, bound_dict in calls.items():
        for shape, bound in bound_dict.items():
            name = f"{key[1]} {[s for s in shape if isinstance(s, tuple)]}"
            results.append(compare(name, ootb[key], ours[key], bound))

    print(json.dumps(dict(margin=MARGIN, gradients=gradients, kernels=results), indent=1))


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 15:02:46.863000 112210 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1019 15:02:46.889000 112210 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an o

ERROR 10-19 15:03:00 [config.py:29] Failed to import Triton kernels. Please make sure your triton version is compatible. Error: No module named 'triton_kernels.matmul_ogs'
LANG_RL Log directory: None


/opt/Megatron-Bridge/3rdparty/Megatron-LM/megatron/core/models/mimo/config/base_configs.py:12: UserWarning: MimoModelConfig is experimental and still under active development. The API may change without notice in future releases.
  class MimoModelConfig:


[gemma4-fast] RMSNorm + value norm: torch.compile
[gemma4-fast] final logit softcapping: torch.compile
[gemma4-fast] attention: sliding layers fa2, global layers te


/opt/Megatron-Bridge/3rdparty/Megatron-LM/megatron/core/transformer/transformer_config.py:2415: UserWarning: full scope is deprecated. Use empty cuda_graph_modules to capture the whole layer.
  warnings.warn(


[gemma4-fast] froze 18 unused k_layernorm weights in KV-shared layers
[gemma4-fast] sliding-window attention: fa2 path in use (seq 1024, head_dim 256, window (511, 0))
[gemma4-fast] global attention: te path in use (seq 1024, head_dim 512, window None)


pip install flash-attn-4==4.0.0b11 nvidia-cutlass-dsl[cu13]


18 parameters frozen; without a gradient: OOTB 18, ours 18: PASS


/opt/venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

_gemma4_rms_norm [(1, 1024, 42, 256), (256,)] PASS  out 1.61e-03 -> 1.61e-03  dhidden_states 2.27e-03 -> 1.67e-03  dweight 1.72e-03 -> 1.72e-03
_gemma4_rms_norm [(1024, 1, 2560), (2560,)] PASS  out 1.64e-03 -> 1.64e-03  dhidden_states 2.29e-03 -> 1.66e-03  dweight 1.70e-03 -> 1.70e-03
_gemma4_rms_norm [(1024, 1, 8, 256), (256,)] PASS  out 1.66e-03 -> 1.66e-03  dhidden_states 2.33e-03 -> 1.66e-03  dweight 1.69e-03 -> 1.69e-03
_gemma4_rms_norm [(1024, 1, 2, 256), (256,)] PASS  out 1.66e-03 -> 1.66e-03  dhidden_states 2.34e-03 -> 1.66e-03  dweight 1.87e-03 -> 1.87e-03
_gemma4_rms_norm [(1024, 1, 8, 512), (512,)] PASS  out 1.66e-03 -> 1.66e-03  dhidden_states 2.33e-03 -> 1.66e-03  dweight 1.52e-03 -> 1.52e-03
_gemma4_rms_norm [(1024, 1, 2, 512), (512,)] PASS  out 1.65e-03 -> 1.65e-03  dhidden_states 2.33e-03 -> 1.66e-03  dweight 1.68e-03 -> 1.68e-03
_logit_softcapping [(1024, 1, 262144)]   PASS  out 2.71e-03 -> 1.66e-03  dlogits 3.08e-03 -> 1.62e-03
_v_norm [(1024, 1, 2, 256)]             